
        DUST WET DEPOSITION SCHEME
        Python reference implementation "The Implementation of a Mineral Dust Wet Deposition Scheme in the GOCART-AFWA Module of the WRF Model"

In [ ]:
"""

This script reproduces the wet deposition formulation described
in the paper.

The scheme separates wet removal into:-  1. In-cloud scavenging  2. Below-cloud scavenging

The main idea is that precipitation removes dust at a rate proportional to the amount of dust already present

The paper uses the following equations

EQUATION (1) - Dust removal

                ∂
                ── = -Λ C
                ∂t

C  = dust concentration
Λ  = scavenging coefficient

This is the basic equation used to calculate the decrease in
dust concentration due to wet deposition.


EQUATION (2) - In-cloud scavenging

                     4.2 × 10^-7 E P
                Λ = ─────────────────
                           dc

E  = collection efficiency (= 0.9 for in-cloud scavenging)
P  = total precipitation [mm h^-1]
dc = mean cloud droplet diameter [µm]

The total precipitation is:

                P = P_grid + P_convective

A grid cell/layer is treated as cloudy when the cloud-water
mixing ratio is greater than zero.

EQUATION (3) - Below-cloud collection efficiency


                         4
                E(dp) = ───── [1 + 0.4 Re^1/2 Sc^1/3
                         Re Sc

                              + 0.16 Re^1/2 Sc^1/2]

                         + 4φ [ μ/μw
                               + φ(1 + 2 Re^1/2) ]

                         + (St - S*)
                           ────────────────────
                           (St - S* + 2/3)

The three terms represent

    First term  -> Brownian diffusion
    Second term -> Interception
    Third term  -> Inertial impaction

where:

                φ = dp / dd

dp = dust particle diameter
dd = mean raindrop diameter

The inertial impaction term is relevant for large particles
(dp > 2 µm) and when:

                St > S*

EQ (4) - Critical Stokes number


                1.2 + ln(1 + Re)/12
        S* = ─────────────────────────
                    1 + ln(1 + Re)

EQ (5) - Reynolds number

                dd Vt(dd) ρa
        Re = ─────────────────
                    2 μ

Re   = Reynolds number of the raindrop
dd   = mean raindrop diameter
Vt   = terminal velocity of the raindrop
ρa   = air density
μ    = kinematic viscosity of air

EQ (6) - Schmidt number


                    μ
        Sc = ───────────────
                  ρa D

Sc = Schmidt number
D  = particle diffusivity


EQ (7) - Stokes number

                    2 τ [Vt(dd) - vt(dp)]
        St = ───────────────────────────────
                              dd

τ     = particle relaxation time
Vt    = raindrop terminal velocity
vt    = dust particle terminal velocity


EQ (8) - Particle diffusivity

                    kb T Cc
        D = ─────────────────────
                3 π μ dp

kb = Boltzmann constant
T  = model temperature
Cc = Cunningham correction factor


EQ (9) - Cunningham correction

                    0.167
        Cc = 1 + ─────────
                      dp


EQ (10) - Particle relaxation time


                    vt
        τ = ─────────
                     g


EQ (11) - Dust particle terminal velocity


                    Cc g
        vt = ─────────────────
                3 π μ dp


EQ (12) - Hydrometeor terminal velocity


                    b
        Vt = a d

For cloud particles:

        a = 3 × 10^7
        b = 2

For rain:

        a = 841.997
        b = 0.8

The paper states that Vt and vt are diagnostically estimated
by the model.



Dust particle effective diameters used in the scheme:

        1.46 µm
        2.80 µm
        4.80 µm
        9.00 µm
        16.0 µm

        
the cloud  water mixing ration > 0 indicte the prsence of cloud and therefor  the incloud scavenging, but it  will come  from the microphysics  morrison2 moment.
"""

'\n\nThis script reproduces the wet deposition formulation described\nin the paper.\n\nThe scheme separates wet removal into:-\n\n    1. In-cloud scavenging\n    2. Below-cloud scavenging\n\nThe main idea is that precipitation removes dust at a rate proportional to the amount of dust already present\n\nThe paper uses the following equations\n\nEQUATION (1) - Dust removal\n\n\n                ∂C\n                ── = -Λ C\n                ∂t\n\nC  = dust concentration\nΛ  = scavenging coefficient\n\nThis is the basic equation used to calculate the decrease in\ndust concentration due to wet deposition.\n\n\nEQUATION (2) - In-cloud scavenging\n\n                     4.2 × 10^-7 E P\n                Λ = ─────────────────\n                           dc\n\nE  = collection efficiency (= 0.9 for in-cloud scavenging)\nP  = total precipitation [mm h^-1]\ndc = mean cloud droplet diameter [µm]\n\nThe total precipitation is:\n\n                P = P_grid + P_convective\n\nA grid cell/layer is treat

In [2]:
import numpy as np

In [3]:
# ============================================================
# CONSTANTS
# ============================================================

# Boltzmann constant, Equation (8)
KB = 1.38066e-23          # J K^-1

# Gravitational acceleration, Equations (10) and (11)
G = 9.81                  # m s^-2


# ============================================================
# DUST SIZE BINS
# ============================================================

# Effective dust particle diameters used by the wet deposition scheme
DUST_DIAMETERS_UM = np.array([
    1.46,
    2.80,
    4.80,
    9.00,
    16.0
])


# ============================================================
# EQ (12) - HYDROMETEOR TERMINAL VELOCITY
# ============================================================

def hydrometeor_terminal_velocity(diameter_um, hydrometeor):
    """
    Calculate hydrometeor terminal velocity using Equation (12):

        Vt = a * d^b

    Cloud:
        a = 3 x 10^7
        b = 2

    Rain:
        a = 841.997
        b = 0.8
    """

    diameter_m = diameter_um * 1.0e-6

    if hydrometeor == "cloud":
        a = 3.0e7
        b = 2.0

    elif hydrometeor == "rain":
        a = 841.997
        b = 0.8

    else:
        raise ValueError("hydrometeor must be 'cloud' or 'rain'")

    Vt = a * diameter_m**b

    return Vt


# ============================================================
# EQ (9) - CUNNINGHAM CORRECTION FACTOR
# ============================================================

def cunningham_correction(dp_um):
    """
    Calculate the Cunningham correction factor.

        Cc = 1 + 0.167 / dp

    dp is used in micrometres, following the paper's
    particle-size parameterization.
    """

    Cc = 1.0 + 0.167 / dp_um

    return Cc


# ============================================================
# EQ (8) - PARTICLE DIFFUSIVITY
# ============================================================

def particle_diffusivity(dp_um, temperature, mu_air):
    """
    Calculate particle diffusivity:

        D = kb T Cc / (3 pi mu dp)
    """

    dp_m = dp_um * 1.0e-6
    Cc = cunningham_correction(dp_um)

    D = (
        KB
        * temperature
        * Cc
        / (3.0 * np.pi * mu_air * dp_m)
    )

    return D


# ============================================================
# EQ (11) - DUST PARTICLE TERMINAL VELOCITY
# ============================================================

def particle_terminal_velocity(dp_um, mu_air):
    """
    Calculate dust particle terminal velocity:

        vt = Cc g / (3 pi mu dp)
    """

    dp_m = dp_um * 1.0e-6
    Cc = cunningham_correction(dp_um)

    vt = (
        Cc
        * G
        / (3.0 * np.pi * mu_air * dp_m)
    )

    return vt


# ============================================================
# EQ (10) - PARTICLE RELAXATION TIME
# ============================================================

def particle_relaxation_time(dp_um, mu_air):
    """
    Calculate particle relaxation time:

        tau = vt / g
    """

    vt = particle_terminal_velocity(dp_um, mu_air)
    tau = vt / G

    return tau


# ============================================================
# EQ (5) - REYNOLDS NUMBER
# ============================================================

def reynolds_number(dd_um, rho_air, mu_air):
    """
    Calculate Reynolds number of the raindrop:

        Re = dd Vt(dd) rho_a / (2 mu)
    """

    dd_m = dd_um * 1.0e-6

    Vt = hydrometeor_terminal_velocity(
        dd_um,
        "rain"
    )

    Re = (
        dd_m
        * Vt
        * rho_air
        / (2.0 * mu_air)
    )

    return Re


# ============================================================
# EQ (6) - SCHMIDT NUMBER
# ============================================================

def schmidt_number(D, rho_air, mu_air):
    """
    Calculate Schmidt number:

        Sc = mu / (rho_a D)
    """

    Sc = mu_air / (rho_air * D)

    return Sc


# ============================================================
# EQ (7) - STOKES NUMBER
# ============================================================

def stokes_number(dp_um, dd_um, mu_air):
    """
    Calculate Stokes number:

        St = 2 tau [Vt(dd) - vt(dp)] / dd
    """

    dd_m = dd_um * 1.0e-6

    Vt_drop = hydrometeor_terminal_velocity(
        dd_um,
        "rain"
    )

    vt_particle = particle_terminal_velocity(
        dp_um,
        mu_air
    )

    tau = particle_relaxation_time(
        dp_um,
        mu_air
    )

    St = (
        2.0
        * tau
        * (Vt_drop - vt_particle)
        / dd_m
    )

    return St


# ============================================================
# EQ (4) - CRITICAL STOKES NUMBER
# ============================================================

def critical_stokes_number(Re):
    """
    Calculate S*:

        S* = [1.2 + ln(1 + Re)/12] / [1 + ln(1 + Re)]
    """

    Sstar = (
        1.2 + np.log(1.0 + Re) / 12.0
    ) / (
        1.0 + np.log(1.0 + Re)
    )

    return Sstar


# ============================================================
# EQ (3) - BROWNIAN DIFFUSION TERM
# ============================================================

def brownian_efficiency(Re, Sc):

    EB = (
        4.0
        / (Re * Sc)
        * (
            1.0
            + 0.4 * Re**0.5 * Sc**(1.0 / 3.0)
            + 0.16 * Re**0.5 * Sc**0.5
        )
    )

    return EB


# ============================================================
# EQ (3) - INTERCEPTION TERM
# ============================================================

def interception_efficiency(
    dp_um,
    dd_um,
    Re,
    mu_air,
    mu_water
):

    phi = dp_um / dd_um

    EI = (
        4.0
        * phi
        * (
            mu_air / mu_water
            + phi * (1.0 + 2.0 * np.sqrt(Re))
        )
    )

    return EI


# ============================================================
# EQ (3) - INERTIAL IMPACTION TERM
# ============================================================

def inertial_efficiency(dp_um, St, Sstar):
    """
    The paper states that inertial impaction is efficient for
    large particles (dp > 2 um) with the restriction St > S*.
    """

    if dp_um > 2.0 and St > Sstar:
        EII = (
            St - Sstar
        ) / (
            St - Sstar + 2.0 / 3.0
        )
    else:
        EII = 0.0

    return EII


# ============================================================
# EQ (3) - TOTAL COLLECTION EFFICIENCY
# ============================================================

def collection_efficiency(
    dp_um,
    dd_um,
    temperature,
    rho_air,
    mu_air,
    mu_water
):
    """
    Calculate all quantities required for the below-cloud
    collection efficiency.
    """

    # Equation (8): particle diffusivity
    D = particle_diffusivity(
        dp_um,
        temperature,
        mu_air
    )

    # Equation (5): Reynolds number
    Re = reynolds_number(
        dd_um,
        rho_air,
        mu_air
    )

    # Equation (6): Schmidt number
    Sc = schmidt_number(
        D,
        rho_air,
        mu_air
    )

    # Equation (7): Stokes number
    St = stokes_number(
        dp_um,
        dd_um,
        mu_air
    )

    # Equation (4): critical Stokes number
    Sstar = critical_stokes_number(Re)

    # Equation (3): Brownian diffusion
    EB = brownian_efficiency(Re, Sc)

    # Equation (3): interception
    EI = interception_efficiency(
        dp_um,
        dd_um,
        Re,
        mu_air,
        mu_water
    )

    # Equation (3): inertial impaction
    EII = inertial_efficiency(
        dp_um,
        St,
        Sstar
    )

    # Equation (3): total collection efficiency
    E = EB + EI + EII

    return {
        "D": D,
        "Re": Re,
        "Sc": Sc,
        "St": St,
        "Sstar": Sstar,
        "EB": EB,
        "EI": EI,
        "EII": EII,
        "E": E
    }


# parameters for wet deposition calculation

def calculate_wet_deposition_parameters(
    dp_um,
    dc_um,
    dd_um,
    temperature,
    rho_air,
    mu_air,
    mu_water
):
  

    # Equation (12): cloud terminal velocity
    Vt_cloud = hydrometeor_terminal_velocity(
        dc_um,
        "cloud"
    )

    # Equation (12): rain terminal velocity
    Vt_rain = hydrometeor_terminal_velocity(
        dd_um,
        "rain"
    )

    # Equation (9): Cunningham correction
    Cc = cunningham_correction(dp_um)

    # Equation (8): particle diffusivity
    D = particle_diffusivity(
        dp_um,
        temperature,
        mu_air
    )

    # Equation (11): dust particle terminal velocity
    vt = particle_terminal_velocity(
        dp_um,
        mu_air
    )

    # Equation (10): relaxation time
    tau = particle_relaxation_time(
        dp_um,
        mu_air
    )

    # Below-cloud quantities from Equations (4)-(7) and (3)
    collection = collection_efficiency(
        dp_um,
        dd_um,
        temperature,
        rho_air,
        mu_air,
        mu_water
    )

    return {
        "Cc": Cc,
        "D": D,
        "vt": vt,
        "tau": tau,
        "Vt_cloud": Vt_cloud,
        "Vt_rain": Vt_rain,
        "Re": collection["Re"],
        "Sc": collection["Sc"],
        "St": collection["St"],
        "Sstar": collection["Sstar"],
        "EB": collection["EB"],
        "EI": collection["EI"],
        "EII": collection["EII"],
        "E": collection["E"]
    }


# ============================================================
# EQ (2) - IN-CLOUD SCAVENGING
# ============================================================

def in_cloud_scavenging(precipitation, dc_um):
    """
    Equation (2):

        Lambda = 4.2 x 10^-7 E P / dc

    E = 0.9 for in-cloud scavenging.
    """

    E = 0.9

    Lambda = (
        4.2e-7
        * E
        * precipitation
        / dc_um
    )

    return Lambda


# ============================================================
# BELOW-CLOUD SCAVENGING
# ============================================================

def below_cloud_scavenging(
    dp_um,
    dd_um,
    precipitation,
    collection
):
    """
    Below-cloud scavenging uses the same form as Equation (2),
    but dd replaces dc and E becomes particle-size dependent.
    """

    E = collection["E"]

    Lambda = (
        4.2e-7
        * E
        * precipitation
        / dd_um
    )

    return Lambda


# ============================================================
# EQ (1) - UPDATE DUST CONCENTRATION
# ============================================================

def update_dust_concentration(C_old, Lambda, dt):
    """
    Equation (1):

        dC/dt = -Lambda C

    Analytical solution:

        C_new = C_old exp(-Lambda dt)
    """

    C_new = C_old * np.exp(-Lambda * dt)

    removed = C_old - C_new

    return C_new, removed


# ============================================================
#  WET DEPOSITION : ACTUAL
# ============================================================

def wet_deposition(
    dp_um,
    dc_um,
    dd_um,
    qcloud,
    P_grid,
    P_convective,
    temperature,
    rho_air,
    mu_air,
    mu_water,
    C_old,
    dt
):
    """
    Calculate actual wet deposition for one grid cell, onemodel layer, and one dust size bin.

    thte seqnce of calculations is:

        1. Calculate all required parameters.
        2. Calculate total precipitation.
        3. Check cloud water mixing ratio.
        4. If qcloud > 0, use in-cloud scavenging.
        5. Otherwise, if precipitation > 0, use below-cloud
           scavenging.
        6. Update dust concentration.
    """

    # STEP 1: calculate all numbers required 

    parameters = calculate_wet_deposition_parameters(
        dp_um,
        dc_um,
        dd_um,
        temperature,
        rho_air,
        mu_air,
        mu_water
    )


    # STEP 2: total precipitation

    P = P_grid + P_convective

  
    # STEP 3: check cloud water mixing ratio FIRST
    #
    # QCLOUD comes from the WRF microphysics scheme.
    # For Morrison 2-moment, QCLOUD is the cloud-water mixing ratio for the model grid cell and model layer.
    
    if qcloud > 0.0:      # incloud scavenging

      

        Lambda = in_cloud_scavenging(
            P,
            dc_um
        )

        regime = "in-cloud"
        E_actual = 0.9

    else:

        if P > 0.0:

      
            # STEP 5: BELOW-CLOUD / RAIN SCAVENGING
 
            Lambda = below_cloud_scavenging(
                dp_um,
                dd_um,
                P,
                parameters
            )

            regime = "below-cloud"
            E_actual = parameters["E"]

        else:

            # No cloud and no precipitation.
            Lambda = 0.0
            regime = "no wet deposition"
            E_actual = 0.0

    # calculate dust removel #

    C_new, removed = update_dust_concentration(
        C_old,
        Lambda,
        dt )

    # Add the wet-deposition result to the calculated parameters
    parameters["qcloud"] = qcloud
    parameters["P_grid"] = P_grid
    parameters["P_convective"] = P_convective
    parameters["P_total"] = P
    parameters["regime"] = regime
    parameters["E_actual"] = E_actual
    parameters["Lambda"] = Lambda
    parameters["C_old"] = C_old
    parameters["C_new"] = C_new
    parameters["removed"] = removed

    return parameters


In [4]:
# TEST CALCULATION


if __name__ == "__main__":

    # test##
    temperature = 280.0       # K
    rho_air = 1.20            # kg m^-3
    mu_air = 1.75e-5          # viscosity used in the equations
    mu_water = 1.0e-3

    # Example precipitation.
    P_grid = 6.0              # mm h^-1
    P_convective = 4.0        # mm h^-1

    # Mean cloud droplet diameter.
    dc = 20.0                 # micrometres

    # Mean raindrop diameter.
    dd = 200.0                # micrometres

    # ---------------------------------------------------------
    #
    #
    # In the WRF run  Cloud water mixing ratio produced by the microphysics scheme Morrison 2-moment.
    
    qcloud = 1.0e-4           # kg kg^-1

    # Initial dust concentration.
    C_initial = 1.0

    # One-hour model timestep.
    dt = 3600.0               # seconds


    # ---------------------------------------------------------
    # CALCULATE WET DEPOSITION
    # ---------------------------------------------------------

    print("WET DEPOSITION")
    print("=" * 80)

    for dp in DUST_DIAMETERS_UM:

        result = wet_deposition(
            dp_um=dp,
            dc_um=dc,
            dd_um=dd,
            qcloud=qcloud,
            P_grid=P_grid,
            P_convective=P_convective,
            temperature=temperature,
            rho_air=rho_air,
            mu_air=mu_air,
            mu_water=mu_water,
            C_old=C_initial,
            dt=dt
        )

        print(
            f"dp = {dp:5.2f} um   "
            f"QCLOUD = {qcloud:.3e}   "
            f"regime = {result['regime']:18s}   "
            f"Lambda = {result['Lambda']:.4e} s^-1   "
            f"C_new = {result['C_new']:.6f}   "
            f"removed = {result['removed']:.6f}"
        )


    # ---------------------------------------------------------
    # SHOW THE CALCULATED PARAMETERS FOR ONE DUST SIZE
    # ---------------------------------------------------------

    dp = 4.8

    result = wet_deposition(
        dp_um=dp,
        dc_um=dc,
        dd_um=dd,
        qcloud=qcloud,
        P_grid=P_grid,
        P_convective=P_convective,
        temperature=temperature,
        rho_air=rho_air,
        mu_air=mu_air,
        mu_water=mu_water,
        C_old=C_initial,
        dt=dt
    )

    print("\nCALCULATED PARAMETERS FOR dp = 4.8 um")
    print("=" * 80)

    print(f"Cc       = {result['Cc']:.6e}")
    print(f"D        = {result['D']:.6e}")
    print(f"vt       = {result['vt']:.6e}")
    print(f"tau      = {result['tau']:.6e}")
    print(f"Vt rain  = {result['Vt_rain']:.6e}")
    print(f"Re       = {result['Re']:.6e}")
    print(f"Sc       = {result['Sc']:.6e}")
    print(f"St       = {result['St']:.6e}")
    print(f"Sstar    = {result['Sstar']:.6e}")
    print(f"EB       = {result['EB']:.6e}")
    print(f"EI       = {result['EI']:.6e}")
    print(f"EII      = {result['EII']:.6e}")
    print(f"E        = {result['E']:.6e}")
    print(f"P total  = {result['P_total']:.6e} mm h^-1")
    print(f"QCLOUD   = {result['qcloud']:.6e} kg kg^-1")
    print(f"Regime   = {result['regime']}")
    print(f"Lambda   = {result['Lambda']:.6e} s^-1")
    print(f"C new    = {result['C_new']:.6e}")
    print(f"Removed  = {result['removed']:.6e}")


WET DEPOSITION
dp =  1.46 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 1.8900e-07 s^-1   C_new = 0.999320   removed = 0.000680
dp =  2.80 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 1.8900e-07 s^-1   C_new = 0.999320   removed = 0.000680
dp =  4.80 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 1.8900e-07 s^-1   C_new = 0.999320   removed = 0.000680
dp =  9.00 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 1.8900e-07 s^-1   C_new = 0.999320   removed = 0.000680
dp = 16.00 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 1.8900e-07 s^-1   C_new = 0.999320   removed = 0.000680

CALCULATED PARAMETERS FOR dp = 4.8 um
Cc       = 1.034792e+00
D        = 5.052977e-12
vt       = 1.282246e+10
tau      = 1.307081e+09
Vt rain  = 9.249847e-01
Re       = 6.342752e+00
Sc       = 2.886088e+06
St       = -1.676000e+23
Sstar    = 4.563372e-01
EB       = 1.811436e-04
EI       = 1.558917e-02
EII      = 0.000000e+